## Time series
something different from cross-sections

Fit AR(p) to y (GDP, say)

In [10]:
# _tsa_00.py
# 코랩 추가.
!pip install yfinance
#!pip install pykrx   # 이건 스크래핑, 무단 자료조회 모듈임. 스킵.
!pip install finance-datareader
!pip install pmdarima

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf                       ## NYSE 데이터 모듈
import FinanceDataReader as fdr             ## KOSPI, NYSE
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭.
from statsmodels.tsa.ar_model import AutoReg
import pmdarima as pm


### KOSPI list, 별도로

In [11]:
# KOSPI 상장 종목 전체 가져오기
#df_kospi = fdr.StockListing('KOSPI')  # 'KOSPI' 이게 변동한 모양. fdr 대신 공공포털 등에서 가져오자.
#df_kospi.head()
#print(df_kospi.head())
#df_kospi = fdr.StockListing('KOSPI-DESC')
#print(df_kospi[['Code', 'Name', 'Sector']].head(7))
# 0  005930  KR7005930003    삼성전자  KOSPI  NaN   257000          2   -24500
# 1  000660  KR7000660001  SK하이닉스  KOSPI  NaN  1688000          2   -42000
# 2  005935  KR7005931001   삼성전자우  KOSPI  NaN   190500          2   -16500
# 3  402340  KR7402340004   SK스퀘어  KOSPI  NaN  1072000          2   -51000
# 4  009150  KR7009150004    삼성전기  KOSPI  NaN



### 코스피 종목, 삼전닉스

In [12]:

# 1. 삼성전자(005930)의 2025년부터 현재까지의 주가 데이터 수집
ticker = '000660'    # SK하이닉스, 티커.
df_dat = fdr.DataReader(ticker, '2023-01-01')  # 삼성전자 코드

# # 2. 가격(종가) 데이터 읽기
# # 최신 버전 FinanceDataReader의 주가 컬럼명은 'Close'입니다.
prc_cls = df_dat['Close']

# # 3. 거래량(볼륨) 데이터 읽기
vlm = df_dat['Volume']

# # 4. 상위 5개 데이터 결합해서 눈으로 확인하기
# print(df_dat[['Close', 'Volume']].head())
# print(df_dat.head(15) )

df_dat.head()


,Open,High,Low,Close,Volume,Change
Date,,,,,,
2023-01-02,75100,76700,75000,75700,1376985,0.009333
2023-01-03,75600,76300,73100,75600,2719437,-0.001321
2023-01-04,75400,81900,75200,81000,5154609,0.071429
2023-01-05,83300,83300,80800,81400,3510964,0.004938
2023-01-06,81400,83600,81100,83100,3687430,0.020885


### differences, percentage changes of a time series

In [13]:
# stck prc over time
yymmdd = df_dat.index   # 자료를 불러올 때, 자동으로 기준 인덱스로 설정된다고 함.
yymmdd = pd.to_datetime( yymmdd )

#yymmdd = df_dat['Date']  이건 안되는 접근.
stck_prc_cls = df_dat['Close']
stck_prc_dff = stck_prc_cls.diff().dropna()
stck_prc_pct = stck_prc_cls.pct_change().dropna()


### autocorrelation at lags = 1 or else.

In [14]:
lag = 2
rho_hat_mat = np.corrcoef(stck_prc_cls[:-lag], stck_prc_cls[lag:])  # :-lag ==> 마지막(lag 만큼) 제외, lag: 첫(lag 만큼)제외
#rho_hat_mat = np.corrcoef(stck_prc_cls[:-1], stck_prc_cls[1:])   # :-1 ==> 마지막 제외, 1: 첫 제외
    #print(rhatmat)
    #np.corrcoef(y[:-k], y[k:])   # 마지막 k, 첫 k 제외.
print(f"시차 상관계수, autocorrelation at lag= {lag} ")
print(rho_hat_mat)

시차 상관계수, autocorrelation at lag= 2 
[[1.         0.99387717]
 [0.99387717 1.        ]]


### Fitting AR(p)
AR(1) model

In [15]:

# 앞에서 살펴볼 변수를 y로 지정해서 이후 코드 그냥 쓰자.
y = stck_prc_dff #cls #pct
# [★해결책] 날짜 인덱스에 영업일(Business Day) 주기 정보를 명시합니다.
# y.index = pd.DatetimeIndex(y.index).to_period('B') # 작동하지만 워닝 나옴. 이런 형식은 제거할 예정이라고.
y = y.asfreq('B').ffill()   # 작동하지만 워닝 나옴.

    # AR(p) model, p = number of lags
model = AutoReg(
        y,
        lags=1,
        trend="c"
        )
result = model.fit()

print( "AR(p) estimates ") # results
print(result.summary() ) #.params)

print(" prediction over 4 periods")
future = result.predict(
        start=len(y),
        end=len(y)+3
        )
print(future)


AR(p) estimates 
                            AutoReg Model Results                             
Dep. Variable:                  Close   No. Observations:                  962
Model:                     AutoReg(1)   Log Likelihood              -11699.813
Method:               Conditional MLE   S.D. of innovations          46895.887
Date:                Wed, 09 Sep 2026   AIC                          23405.627
Time:                        00:02:06   BIC                          23420.231
Sample:                    01-04-2023   HQIC                         23411.188
                         - 09-09-2026                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       2074.2047   1513.998      1.370      0.171    -893.177    5041.587
Close.L1      -0.0844      0.032     -2.626      0.009      -0.147      -0.021
                                   

### Fitting y to ARIMA(p,d,q)
ARIMA(1,1,1)

In [16]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# 1. 종가 데이터 추출
stck_prc_cls = df_dat['Close']

# 2. 영업일(B) 주기 설정 및 공휴일 결측치 채우기
# ARIMA는 원본 데이터 자체에서 차분(d)을 수행할 수 있으므로, 원본 종가를 그대로 사용합니다.
y = stck_prc_cls.asfreq('B', method='ffill')

# 3. ARIMA(p, d, q) 모델 설정
# order=(p, d, q) 구조입니다.
# 예시: order=(1, 1, 1) -> AR(1), 1차 차분, MA(1)
model = ARIMA(
    y,
    order=(1, 1, 1),      # 원하시는 p, d, q 숫자를 여기에 넣으세요
    trend='n'             # 'c' 상수항(intercept) 포함. 't' 시간추세, 'ct' 상수항과 시간 추세, 'n' 없을 때.
)

# 4. 모델 피팅 (Model Fitting)
result = model.fit()

# 5. 결과 요약 출력
print(result.summary())

# 6. 미래 4개 시점 예측 (Forecast)
# ARIMA에서는 predict 대신 forecast() 함수를 쓰면 미래 시점을 지정하기 훨씬 편합니다.
future = result.forecast(steps=4)
print("\n--- 미래 4개 시점 예측값 ---")
print(future)


                               SARIMAX Results                                
Dep. Variable:                  Close   No. Observations:                  963
Model:                 ARIMA(1, 1, 1)   Log Likelihood              -11679.456
Date:                Wed, 09 Sep 2026   AIC                          23364.913
Time:                        00:02:06   BIC                          23379.520
Sample:                    01-02-2023   HQIC                         23370.475
                         - 09-09-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.0454      0.068      0.665      0.506      -0.088       0.179
ma.L1         -0.2189      0.062     -3.506      0.000      -0.341      -0.097
sigma2      2.073e+09   1.07e-10   1.93e+19      0.0

### Automatic selection of ARIMA(p,d,q)

auto_arima 최종 결과가 (p,d,q) = (2,1,0)인 경우.

1. 직관적인 차분 형태
$$
\Delta y_t = c + \phi_1 \Delta y_{t-1} + \phi_2 \Delta y_{t-2} + \varepsilon_t
$$

2. 연산자(B)를 활용한 정석 형태
$$
(1 - \phi_1 B - \phi_2 B^2)(1 - B)y_t = c + \varepsilon_t
$$

In [17]:
import pandas as pd
import pmdarima as pm

# 1. 주가 데이터 가져오기 및 주기 설정
# auto_arima 역시 결측치가 있으면 돌아가지 않으므로 ffill 처리를 해줍니다.
y = df_dat['Close'].asfreq('B', method='ffill')

# 2. 최적의 ARIMA 모델 자동 탐색 및 학습 (Auto Fit)
# 원본 데이터를 그대로 넣으면 최적의 차분(d) 횟수까지 알아서 계산합니다.
auto_model = pm.auto_arima(
    y,
    start_p=1, max_p=5,       # AR(p) 탐색 범위 (1부터 5까지)
    start_q=1, max_q=5,       # MA(q) 탐색 범위 (1부터 5까지)
    d=None,                   # None으로 두면 데이터가 안정적일 때까지 알아서 차분 횟수를 정함
    seasonal=False,           # 주가 데이터는 보통 계절성이 없으므로 False
    trace=True,               # 어떤 조합들을 테스트하고 있는지 콘솔에 출력하는 옵션
    error_action='ignore',    # 에러가 나는 조합(아까 보셨던 trend 에러 등)은 알아서 패스
    suppress_warnings=True,   # 자잘한 경고창 끄기
    stepwise=True             # 스마트 탐색 알고리즘을 써서 속도를 획기적으로 단축
)

# 3. 인공지능이 찾아낸 최적의 차수 확인하기
print("\n--- 최적의 모델 차수 결정 결과 ---")
print(auto_model.summary())
# 4. 미래 4개 시점 예측하기
# pmdarima의 예측 함수는 내장 함수 이름이 predict입니다.
future = auto_model.predict(n_periods=4)
print("\n--- auto_arima 미래 4개 시점 예측값 ---")
print(future)

# 기존 auto_arima 학습 코드 아래에 추가하세요
print("--- [확인] 모델 핵심 계수(값) 목록 ---")
print(auto_model.params())
# 기존 auto_arima 학습 코드 아래에 추가하세요

# 내부 statsmodels 결과 객체에서 계수들만 쏙 뽑아 딕셔너리로 변환합니다.
params_dict = auto_model.arima_res_.params.to_dict()

print("\n====== [최종 계수 값] ======")
for key, value in params_dict.items():
    print(f"{key} ===> {value:.4f}")
print("============================")



Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=23364.714, Time=0.15 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=23390.029, Time=0.05 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=23367.305, Time=0.06 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=23362.923, Time=0.09 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=23389.507, Time=0.08 sec
 ARIMA(0,1,2)(0,0,0)[0] intercept   : AIC=23364.091, Time=0.30 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=23360.049, Time=0.64 sec
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=23291.440, Time=3.70 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=23349.849, Time=0.64 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=23301.045, Time=1.38 sec
 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=23301.785, Time=1.60 sec
 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=23331.680, Time=0.68 sec
 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=23327.480, Time=0.53 sec
 ARIMA(3,1,3)(0,0,0)[0] intercept   : AIC=23302.169, Time=0.77 sec
 ARIMA(2,1,2)(0,0,0